## Notebook to demo how to create a feature class from a parquet file.

```
pip install duckdb fastparquet
```

In [1]:
import os
import arcpy
import duckdb
import pandas as pd
import pyarrow as pa

In [2]:
base = os.path.join("Z:", os.sep, "geo_addr.prq")
path = os.path.join(base, "*.parquet")

In [3]:
os.path.exists(base)

True

In [4]:
#conn = duckdb.connect(":memory:")

In [5]:
#_ = conn.execute("INSTALL spatial;LOAD spatial;")

In [6]:
#pdf = conn.sql(f"""
#select * from read_parquet("{path}");
#""").df()

In [7]:
pdf = pd.read_parquet(base, engine="fastparquet")

In [8]:
sp_ref = arcpy.SpatialReference(2236)
metadata = {"esri.encoding": "WKB", "esri.sr_wkt": sp_ref.exportToString()}
pa_shape = pa.field("SHAPE",
                    pa.binary(),
                    nullable=False,
                    metadata=metadata)

In [9]:
feature_class_name = "GeoAddr"
workspace = "memory" # arcpy.env.scratchGDB

schema = pa.Schema.from_pandas(pdf)
shape_index = schema.get_field_index("SHAPE")
schema = schema.set(shape_index, pa_shape)
tab = pa.Table.from_pandas(pdf, schema=schema)

fc = os.path.join(workspace, feature_class_name)
arcpy.management.Delete(fc)
arcpy.management.CopyFeatures(tab, fc)

<Result 'memory\\GeoAddr'>